## Imports
Run the cell below first. It loads every library used in this notebook:
- **General:** `math` (rounding), `webbrowser` and `pathlib.Path` (opening/locating saved HTML files)
- **Data:** `pandas` (DataFrames, merging, resampling) and `numpy` (numeric helpers, `np.where`)
- **Static plots:** `seaborn` and `matplotlib.pyplot`
- **Bokeh:** `figure`, `show`, `output_file`, `save`, `ColumnDataSource`, `HoverTool`, `Slider`, `CustomJS`, `column`, `DatetimeTickFormatter`, `DaysTicker` (interactive plots, sliders, time axes)
- **Plotly:** `plotly.express` (animated and marginal scatter plots)
- **Altair:** `altair` (declarative interactive charts)
- **Statistics:** `statsmodels.tsa.stattools.acf` (autocorrelation)

In [ ]:
import math
import webbrowser
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import altair as alt

from bokeh.io import output_file, save
from bokeh.layouts import column
from bokeh.models import (
    ColumnDataSource,
    CustomJS,
    DatetimeTickFormatter,
    DaysTicker,
    HoverTool,
    Slider,
)
from bokeh.plotting import figure, show

from statsmodels.tsa.stattools import acf

## Practice 1
Diamonds dataset: create flag columns with `np.where`, scatter plot of carat vs. price, and `apply` a rounding function to price.

In [ ]:
diamonds_url = (
    "https://raw.githubusercontent.com/TrainingByPackt/"
    "Interactive-Data-Visualization-with-Python/master/datasets/diamonds.csv"
)
diamonds_df = sns.load_dataset('diamonds')
try:
    diamonds_df = pd.read_csv(diamonds_url)
    print("[OK] Dataset successfully loaded from URL.\n")
except Exception as e:
    print(f"[Warning] Could not load from URL due to: {e}")
    print("Loading from Seaborn instead...\n")
    diamonds_df = sns.load_dataset('diamonds')

diamonds_df["large_diamond"] = np.where(diamonds_df["carat"] > 1, 1, 0)
print(diamonds_df.head())
print(diamonds_df['large_diamond'].value_counts())
sns.scatterplot(
    data = diamonds_df,
    x="carat",
    y="price",
    hue="cut",
    alpha = 0.5
)

mask = (diamonds_df["cut"] == "Premium") & (diamonds_df["color"] == "E")
diamonds_df["premium_diamond"] = np.where(mask, 1, 0)
print(diamonds_df.head())
print(diamonds_df['premium_diamond'].value_counts())

def round_up(price):
    return math.ceil(price/100)*100

diamonds_df["rounded_price_100"] = diamonds_df["price"].apply(round_up)
print(diamonds_df[["price", "rounded_price_100"]].head())
diamonds_df = diamonds_df.drop("rounded_price_100", axis=1)
print(diamonds_df.head())
plt.show()



## Practice 2 – Problem 1
Diamonds: histogram + KDE of prices, bar plot of mean price by cut and color, and annotated count plot of cuts.

In [ ]:
diamonds_url = (
    "https://raw.githubusercontent.com/TrainingByPackt/"
    "Interactive-Data-Visualization-with-Python/master/datasets/diamonds.csv"
)

diamonds_df = sns.load_dataset('diamonds')

sns.histplot(
    data = diamonds_df,
    x = "price",
    bins = 50,
    kde = True
)

plt.title("Disribution of Diamond Prices")
plt.xlabel("Price (USD)")
plt.ylabel("Count")
plt.show()
color_list = ['D','E','F','G','H','J']

sns.barplot(
    data = diamonds_df,
    x = "cut",
    y = "price",
    hue_order = color_list,
    hue = "color"

)

plt.title("Mean Diamod Price by Cut and Color")
plt.ylabel("Mean Price (USD)")
plt.xlabel("Diamond Cut")
plt.legend(loc="upper right", bbox_to_anchor=(1.15,1.0))
sns.despine()
plt.tight_layout()
plt.show()

plt.figure() #graph 1

g1 = sns.countplot(
    data = diamonds_df,
    x = "cut"
)
ax1 = g1

for container in ax1.containers:
    ax1.bar_label(container, fmt = ("%.0f"), fontsize = 8)

ideal_count = (diamonds_df["cut"] == "Ideal").sum()
x_pos = [t.get_text() for t in ax1.get_xticklabels()].index("Ideal")

ymin, ymax = ax1.get_ylim()
needed_top = ideal_count + 5000
if ymax < needed_top:
    ax1.set_ylim(ymin, needed_top)

ax1.annotate(
    "Most Common Cut",
    xy = (x_pos, ideal_count),
    xytext = (8,25),
    textcoords = "offset points",
    ha = "left", va = "bottom",
    fontsize = 10,
    bbox = dict(
        boxstyle = "round, pad = 0.35",
        facecolor = "white",
        edgecolor = "0.7",
        linewidth = 1
    ),
    arrowprops = dict(
        arrowstyle = "->",
        color = "crimson",
        linewidth = 2,
        shrinkA = 0, shrinkB = 0,
        connectionstyle = "arc3, rad = 0"
    ),
    zorder = 10,
    clip_on = False
)
plt.title("Diamonds Count by Cut")
plt.xlabel("Diamond Cut")
plt.ylabel("Count")
plt.tight_layout()

## Practice 2 – Problem 2
Olympics 2016: top 5 sports by medal count (count plot).

In [ ]:
#Practice 2
URL = "https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/athlete_events.csv"
olympics_df = pd.read_csv(URL)

olympics_winners = olympics_df.dropna(subset=['Medal']).copy()

YEAR = 2016
winners_2016 = olympics_winners.loc[olympics_winners["Year"] == YEAR].copy()

top5_2016 = winners_2016["Sport"].value_counts().head(5)
print("[Counts] Medals by sport in 2016 (top 5):")
print(top5_2016, "\n")

top5_order = top5_2016.index
plt.figure() # second graph
g2 = sns.countplot(
    data = winners_2016,
    order = top5_order,
    x = "Sport",
    legend = False
)

for container in g2.containers:
    g2.bar_label(container, fmt = ("%.0f"), fontsize = 8, padding = 2)

plt.title("Top 5 Sports by Medal Count - 2016")
plt.xlabel("Sport")
plt.ylabel("Medal Count")
plt.tight_layout()
plt.show()


## Practice 3 – Problem 1
MPG dataset: histogram of vehicle weight with mean and median lines.

In [ ]:
mpg_df = sns.load_dataset("mpg")

#Graph 1
plt.figure()
sns.histplot(
    data = mpg_df.weight,
    bins = 40,
    kde = False
)
plt.axvline(
    x = np.median(mpg_df.weight),
    color = 'orange',
    label = "Median"
)
plt.axvline(
    x = np.mean(mpg_df.weight),
    color = 'red',
    label = "Mean"
)
plt.title("Distribution of Vehicle Weight")
plt.xlabel("Weight")
plt.ylabel("Frequency")
plt.legend(loc = 'upper right')

## Practice 3 – Problem 2
MPG dataset: box plot of MPG by number of cylinders.

In [ ]:
#Graph 2
mpg_clean = mpg_df.dropna(subset=['mpg', 'cylinders']).copy()
plt.figure()
sns.boxplot(
    data = mpg_clean,
    x = 'cylinders',
    y = 'mpg',
    boxprops = {'edgecolor': 'red'},
    whiskerprops = {'color': 'red'},
    capprops = {'color': 'red'}
)
plt.title("MPG by Number of Cylinders")
plt.xlabel("Number of Cylinders")
plt.ylabel("MPG")

plt.show()


## Practice 4 – Problem 1
MPG dataset: violin plot of MPG by decade.

In [ ]:
mpg_df = sns.load_dataset("mpg")

mpg_clean = mpg_df.dropna(subset=['mpg', 'model_year']).copy()
mpg_clean['decade'] = pd.cut(
    mpg_clean['model_year'],
    bins = [69,79,89],
    labels = ["1970s", '1980s']
)
plt.figure()
sns.violinplot(
    data = mpg_clean,
    x = 'decade',
    y = 'mpg',
    inner = 'quart'
)
plt.title("MPG Distribution by Decade")
plt.xlabel("Decade")
plt.ylabel("Miles Per Gallon (MPG)")
plt.tight_layout()

## Practice 4 – Problem 2
MPG dataset: violin plot of MPG by decade and origin with a strip plot overlay.

In [ ]:
plt.figure()
sns.violinplot(
    data = mpg_clean,
    x = 'decade',
    y = 'mpg',
    hue = 'origin',
    inner = 'quart',
    split = False
)
sns.stripplot(
    data = mpg_clean,
    x = 'decade',
    y = 'mpg',
    hue = 'origin',
    dodge = True,
    jitter = True,
    palette = ["white", "white", "white"],
    edgecolor = 'red',
    linewidth = 1,
    legend = False
)
plt.title("MPG Distribution by Decade and Origin")
plt.xlabel("Decade")
plt.ylabel("Miles Per Gallon (MPG)")
plt.tight_layout()
plt.show()




## Practice 5 – Problem 1
Olympics 2016 medal winners: violin plot of age by medal type and sex.

In [ ]:
dataset_url = "https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/athlete_events.csv"
olympics_df = pd.read_csv(dataset_url)
selected_sports = ['Athletics', 'Swimming', 'Rowing', 'Football', 'Hockey']
olympics_2016 = olympics_df[(olympics_df['Year'] == 2016) &
                            (olympics_df['Sport'].isin(selected_sports)) &
                            (olympics_df['Medal'].notna())].copy()
olympics_clean = olympics_2016.dropna(subset=['Age']).copy()

plt.figure()
sns.violinplot(
    data = olympics_clean,
    x = 'Medal',
    y = 'Age',
    hue = 'Sex',
    palette = 'Set2',
    inner = 'quart',
    order = ['Bronze', "Silver", 'Gold']
)
plt.title("Age Distribution by Medal Type and Gender")
plt.xlabel("Medal Type")
plt.ylabel("Age")
plt.tight_layout()

## Practice 5 – Problem 2
Olympics 2016 medal winners: pair plot of height, weight and age.

In [ ]:
olympics_clean = olympics_2016.dropna(subset=['Height', 'Weight', 'Age']).copy()
g = sns.pairplot(
    data = olympics_clean,
    vars = ['Height', 'Weight', 'Age'],
    hue = 'Medal',
    palette = 'Set2',
    corner = True,
    diag_kind = 'hist',
    plot_kws = {'alpha': 0.7, 's': 40, 'edgecolor': 'none'},
    diag_kws = {'bins': 15, 'alpha': 0.7, 'linewidth': 0.5}
)
g.figure.suptitle("Pairwise Relationships: Height, Weight, and Age\n2016 Medal Winners", y = 1)
g.figure.subplots_adjust(top=0.90)
g.figure.tight_layout()
plt.show()


## Practice 6
Reshape wide CO₂ data to long format with `melt`, convert types, drop missing values, and sort.

In [ ]:
co2_wide = pd.DataFrame({
    'country': ['Canada', 'Mexico', 'Japan'],
    'region' : [
        'North America',
        'Latin America',
        'East Asia'
    ],
    '2020': [14.2, 3.7, 8.4],
    '2021': [14.5, 'missing', 8.1]
})

co2_long = pd.melt(
    co2_wide,
    id_vars = ['country', 'region'],
    var_name = 'year',
    value_name = 'co2'
)
co2_long['year'] = pd.to_numeric(co2_long['year'], errors = 'coerce')
co2_long['co2'] = pd.to_numeric(co2_long['co2'], errors = 'coerce')
co2_long = co2_long.dropna(subset = ['co2'])
co2_long['year'] = co2_long['year'].astype('int64')
co2_long_clean = co2_long.sort_values(by = ['country', 'year']).reset_index(drop = True)

print("Original wide-format data:\n", co2_wide)
print('\nLong-format data:\n',co2_long)
print('\nCleaned CO₂ data:\n', co2_long_clean)


## Practice 7 – Problem 1
Merge CO₂ and Gapminder data and filter to a single year (1964).

In [ ]:
URL_CO2 = ("https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/co2.csv")
URL_GM = ("https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/gapminder.csv")
co2 = pd.read_csv(URL_CO2, keep_default_na=True)
gm = pd.read_csv(URL_GM, keep_default_na=True)

regions = gm[["Country", "region"]].dropna().drop_duplicates()
merged = (
    co2.merge(
        regions,
        left_on="country",
        right_on="Country",
        how="inner"
    )
    .drop(columns=["Country"])
)
year_cols = [c for c in merged.columns if str(c).isdigit() and len(str(c)) == 4]
new_co2 = merged.melt(
    id_vars=["country", "region"],
    value_vars=year_cols,
    var_name="year",
    value_name="co2"
)

# Convert year and CO2 values to numeric
new_co2["year"] = pd.to_numeric(new_co2["year"], errors="coerce")
new_co2["co2"] = pd.to_numeric(new_co2["co2"], errors="coerce")
df_gdp = gm[["Country", "Year", "gdp"]].rename(columns={"Country": "country", "Year": "year"}).copy()
df_gdp["year"] = pd.to_numeric(df_gdp["year"], errors="coerce")
df_gdp["gdp"] = pd.to_numeric(df_gdp["gdp"], errors="coerce")
df_gdp = (df_gdp.dropna(subset=["year", "gdp"]).drop_duplicates(["country", "year"]))
new_co2 = new_co2.dropna(subset=["year"])
new_co2["year"] = new_co2["year"].astype("int64")
df_gdp["year"] = df_gdp["year"].astype("int64")

data = new_co2.merge(
    df_gdp,
    on=["country", "year"],
    how="inner"
)

selected_year = 1964
# 1. Keep only rows for the selected year
year_df = data[data["year"] == selected_year]
# 2. Remove rows where GDP or CO2 is missing
year_df = year_df.dropna(subset=['gdp', 'co2'])

print("Data loaded successfully.")
print("Columns:", data.columns.tolist())
print()
print(year_df.head())

## Practice 7 – Problem 2
Bokeh interactive scatter plot of CO₂ vs. GDP with log y-axis and hover tooltips.

In [ ]:
#Practice 2
year_df = (data.loc[data['year'] == selected_year, ['country','region', 'gdp', 'co2']]).dropna(subset = ['gdp','co2'])
year_df = year_df[year_df['co2'] > 0].copy()

source = ColumnDataSource(data = {
    'x': year_df['gdp'].tolist(),
    'y': year_df['co2'].tolist(),
    'country': year_df['country'].tolist(),
    'region': year_df['region'].tolist()
})
plot = figure(
    title="CO2 Emissions vs GDP in 1964",
    width=800,
    height=500,
    x_axis_label="GDP per person",
    y_axis_label="CO2 emissions per person",
    y_axis_type="log",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover"
)
plot.scatter(
    x = 'x',
    y = 'y',
    source = source,
    size = 8,
    fill_alpha = 0.7
)

hover = plot.select_one(HoverTool)
hover.tooltips = [
    ('Country', "@country"),
    ('Region', "@region"),
    ('GDP', "@x{0,0}"),
    ('CO2', "@y{0.00}")
]

print("Data loaded successfully.")
print("Columns:", data.columns.tolist())
print()
show(plot)

#Optional
#log grows on the y levels fast, similar to a logarithmic graph
#linear graph slips the graph, as it grows less on the y axis and more on the x axis


## Practice 8 – Problem 1
Filter a small DataFrame to a selected year.

In [ ]:
data = pd.DataFrame({
    "country": ["USA", "China", "India","USA", "China", "India"],
    "region": ["America", "Asia", "Asia","America", "Asia", "Asia"],
    "year": [1990, 1990, 1990,2000, 2000, 2000],
    "gdp": [24000, 1000, 600,36000, 2200, 900],
    "co2": [19.0, 2.1, 0.8,20.5, 2.7, 1.0]
})

selected_year = 2000
year_df = data['year'] == selected_year
selected_data = data.loc[year_df, ['country', 'gdp', 'co2']]
print(selected_data)

## Practice 8 – Problem 2
Bokeh scatter plot of CO₂ vs. GDP with a year slider (`CustomJS`), saved to HTML.

In [ ]:
data = pd.DataFrame({
    "country": [
        # 1990
        "USA", "China", "India", "Germany", "Brazil", "Japan",
        # 1995
        "USA", "China", "India", "Germany", "Brazil", "Japan",
        # 2000
        "USA", "China", "India", "Germany", "Brazil", "Japan",
        # 2005
        "USA", "China", "India", "Germany", "Brazil", "Japan",
        # 2010
        "USA", "China", "India", "Germany", "Brazil", "Japan",
        # 2015
        "USA", "China", "India", "Germany", "Brazil", "Japan"
    ],
    "year": [
        1990, 1990, 1990, 1990, 1990, 1990,
        1995, 1995, 1995, 1995, 1995, 1995,
        2000, 2000, 2000, 2000, 2000, 2000,
        2005, 2005, 2005, 2005, 2005, 2005,
        2010, 2010, 2010, 2010, 2010, 2010,
        2015, 2015, 2015, 2015, 2015, 2015
    ],
    "gdp": [
        # 1990
        24000, 1000, 600, 22000, 3000, 25000,
        # 1995
        28000, 1400, 700, 26000, 4200, 29000,
        # 2000
        36000, 2200, 900, 30000, 4800, 34000,
        # 2005
        44000, 3500, 1200, 35000, 6000, 39000,
        # 2010
        49000, 6000, 1600, 41000, 8500, 44000,
        # 2015
        57000, 8000, 2100, 46000, 9000, 39000
    ],
    "co2": [
        # 1990
        19.0, 2.1, 0.8, 12.0, 1.4, 9.5,
        # 1995
        19.5, 2.5, 0.9, 11.0, 1.6, 9.8,
        # 2000
        20.5, 2.7, 1.0, 10.5, 1.8, 9.7,
        # 2005
        19.5, 4.2, 1.2, 10.0, 1.9, 9.5,
        # 2010
        17.5, 6.5, 1.5, 9.2, 2.1, 9.0,
        # 2015
        15.5, 7.5, 1.8, 8.8, 2.2, 8.5
    ]
})

all_source = ColumnDataSource(data = {
    'year': data['year'].tolist(),
    'gdp': data['gdp'].tolist(),
    'co2': data['co2'].tolist()
})
year_df = data[data['year'] == 1990]
source = ColumnDataSource(data = {
    'x': year_df['gdp'].tolist(),
    'y': year_df['co2'].tolist()
})

plot = figure(
    title = 'CO2 vs GDP in 1990',
    width = 700,
    height = 400,
    x_axis_label = 'GDP per capita',
    y_axis_label = 'CO2 per person'
)
plot.scatter(
    x = 'x',
    y = 'y',
    source = source,
    size = 10
)

slider = Slider(
    start = 1990,
    end = 2015,
    step = 5,
    value = 1990,
    title = 'Year'
)

callback = CustomJS(
    args = dict(
        source = source,
        all_source = all_source,
        slider = slider,
        plot = plot
    ),
    code = """
    const yr = slider.value;
    const A = all_source.data;
    const x = [];
    const y = [];

    for(let i = 0; i < A.year.length; i++)
    {
        if(A.year[i] == yr)
        {
            x.push(all_source.data["gdp"][i]);
            y.push(all_source.data["co2"][i]);
        }
    }

    source.data = {x,y}
    source.change.emit();

    plot.title.text = 'CO₂ Emissions vs GDP in ' + yr;
    """)

slider.js_on_change('value', callback)
layout = column(slider, plot)
output_file('excercise2.html')
save(layout)
print("Interactive visualization saved as excercise2.html")


## Practice 9
Plotly Express: merge CO₂ and Gapminder data and build an animated CO₂ vs. GDP scatter plot by year, colored by region.

In [ ]:
co2 = pd.read_csv('co2.csv')
gm = pd.read_csv('gapminder.csv')
df_gm = gm[['Country', 'region']].drop_duplicates()

df_w_regions = pd.merge(
    co2,
    df_gm,
    left_on = 'country',
    right_on = 'Country',
    how = 'inner'
)
df_w_regions = df_w_regions.drop('Country', axis = 'columns')

new_co2 = pd.melt(
    df_w_regions,
    id_vars = ['country', 'region'],
    var_name = 'year',
    value_name = 'co2'
)

new_co2['year'] = pd.to_numeric(
    new_co2['year'],
    errors = 'coerce'
)

df_co2 = new_co2[new_co2['year'] >= 1964].copy()
df_co2 = df_co2.sort_values(
    by = ['country', 'year']
)
print(df_co2.head(10))

new_co2['co2'] = pd.to_numeric(
    new_co2['co2'],
    errors = 'coerce'
)

df_co2 = new_co2.dropna(subset = ['year', 'co2']).copy()
df_co2 = df_co2[df_co2['year'] >= 1964]
df_co2 ['year'] = df_co2['year'].astype('int64')

df_gdp = gm[['Country', 'Year', 'gdp']].copy()
df_gdp.columns = [
    'country',
    'year',
    'gdp'
]

data = pd.merge(
    df_co2,
    df_gdp,
    on = ['country', 'year'],
    how = 'left'
)

data['gdp'] = pd.to_numeric(
    data['gdp'],
    errors = 'coerce'
)
data = data.dropna(subset = ['co2', 'gdp', 'region'])
data = data[data['gdp'] > 0]

fig = px.scatter(
    data,
    x = 'gdp',
    y = 'co2',
    color = 'region',
    hover_name = 'country',
    animation_frame = 'year',
    animation_group = 'country',
    log_x = True
)

fig.write_html(
    "exercise2_plot.html",
    include_plotlyjs=True
)

fig.show()


## Practice 10 – Problem 1
Plotly Express: CO₂ by year scatter plot with marginal box plot.

In [ ]:
url_co2 = ("https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/co2.csv")
url_gm = ("https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/gapminder.csv")
co2 = pd.read_csv(url_co2)
gm = pd.read_csv(url_gm)

df_gm = gm[['Country', 'region']].drop_duplicates()
df_w_regions = pd.merge(
    co2,
    df_gm,
    left_on = 'country',
    right_on = 'Country',
    how = 'inner'
)
df_w_regions = df_w_regions.drop('Country', axis = 'columns')

new_co2 = pd.melt(
    df_w_regions,
    id_vars = ['country', 'region'],
    var_name = 'year',
    value_name = 'co2'
)
new_co2['year'] = new_co2['year'].astype('int64')

df_co2 = new_co2[new_co2['year'] >= 1964]
df_co2 = df_co2.sort_values(by = ['country', 'year'])
df_gdp = gm[['Country', 'Year', 'gdp']].copy()
df_gdp.columns = ['country', 'year', 'gdp']

data = pd.merge(
    df_co2,
    df_gdp,
    on = ['country','year'],
    how = 'inner'
)
data['co2'] = pd.to_numeric(
    data['co2'],
    errors = 'coerce'
)
data['gdp'] = pd.to_numeric(
    data['gdp'],
    errors = 'coerce'
)
data = data.dropna(subset = ['co2', 'gdp'])

fig = px.scatter(
    data,
    x = 'year',
    y = 'co2',
    color = 'region',
    marginal_y = 'box',
    hover_name = 'country',
    hover_data = {
        'region': True,
        'year': True,
        'co2': ':.2f'
    }
)
fig.write_html("exercise1_co2_year.html")
fig.show()

## Practice 10 – Problem 2
Plotly Express: animated CO₂ vs. GDP scatter with marginal plots and redraw fix.

In [ ]:
data2 = pd.merge(
    df_co2,
    df_gdp,
    on = ['country','year'],
    how = 'left'
)
data2['co2'] = pd.to_numeric(
    data['co2'],
    errors = 'coerce'
)
data2['gdp'] = pd.to_numeric(
    data['gdp'],
    errors = 'coerce'
)
data2 = data.dropna(subset = ['co2', 'gdp'])

#fix marginal graphs not updated with animation (semi fix, part 1/3):
x_pad = (data2['gdp'].max() - data2['gdp'].min()) * 0.05
y_pad = (data2['co2'].max() - data2['co2'].min()) * 0.05

fig = px.scatter(
    data2,
    x = 'gdp',
    y = 'co2',
    color = 'region',
    marginal_x = 'rug',
    marginal_y = 'box',
    hover_name = 'country',
    animation_frame = 'year',
    animation_group = 'country',
    
    #fix marginal graphs not updated with animation (semi fix, part 2/3):
    range_x = [data2['gdp'].min() - x_pad, data2['gdp'].max() + x_pad],
    range_y = [data2['co2'].min() - y_pad, data2['co2'].max() + y_pad]
)

#fix marginal graphs not updated with animation (semi fix, part 3/3):
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['redraw'] = True
for step in fig.layout.sliders[0].steps:
    step.args[1]['frame']['redraw'] = True

fig.write_html(" exercise2_gdp_co2_animated.html")
fig.show()


## Practice 11 – Problem 1
Altair: HPI scatter plot with a minimum-wellbeing slider filter.

In [ ]:
hpi_url = ("https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/hpi_data_countries.tsv")
hpi_df = pd.read_csv(hpi_url, sep = '\t')
alt.renderers.enable('default')

wb_slider = alt.param(
    name = 'wb_min',
    value = 5.0,
    bind = alt.binding_range(min = 0.0, max = 10.0, step = 0.1, name = "min Wellbeing: ")
)

chart = (
    alt.Chart(hpi_df, width = 700, height = 500)
    .mark_circle()
    .encode(
        x = 'Wellbeing (0-10)',
        y= 'Happy Planet Index',
        color = 'Region:N',
        size = alt.Size(
            'Ecological Footprint (gha/capita):Q',
            scale = alt.Scale(range = [50,1000]),
            title = 'Ecological Footprint (gha/capita)'
        ),
        tooltip = [
            'Country:N', 
            'Region:N', 
            'Wellbeing (0-10):Q', 
            'Happy Planet Index:Q'
        ]
    )
    .add_params(wb_slider)
    .transform_filter((alt.datum["Wellbeing (0-10)"] >= wb_slider))
    .interactive()
)

chart.show()
chart.save('lec15_ex01.html')
print("Saved lec15_ex01.html")

## Practice 11 – Problem 2
Altair: HPI scatter plot with a region dropdown filter.

In [ ]:
#Exercise 2

region_values = sorted(hpi_df["Region"].dropna().unique().tolist())
region_options = ['All'] + region_values

region_select = alt.param(
    name = 'region_pick',
    value = 'All',
    bind = alt.binding_select(options = region_options, name = 'Region: ')
)

chart = (
    alt.Chart(hpi_df, width = 900, height = 600)
    .mark_circle()
    .add_params(region_select)
    .encode(
        x = 'Wellbeing (0-10)',
        y= 'Happy Planet Index',
        color = 'Region',
        size = alt.Size(
                    'Ecological Footprint (gha/capita):Q',
                    scale = alt.Scale(range = [50,1000]),
                    title = 'Ecological Footprint (gha/capita)'
                ),
        tooltip = [
                    'Country:N', 
                    'Region:N', 
                    'Wellbeing (0-10):Q', 
                    'Happy Planet Index:Q'
                ]
    )
    .transform_filter((region_select == 'All') | (alt.datum.Region == region_select))
    .interactive()
)

chart.show()
chart.save('lec15_ex02.html')
print("Saved lec15_ex02.html")


## Practice 12 – Problem 1
Altair: HPI scatter plot highlighting a region chosen from a dropdown.

In [ ]:
hpi_url = "https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/hpi_data_countries.tsv"
hpi_df = pd.read_csv(hpi_url, sep = '\t')

regions = sorted(hpi_df['Region'].dropna().unique().tolist())
region_options = ['All'] + regions
region_pick = alt.param(
    name = 'Select',
    value = 'All',
    bind = alt.binding_select(
        options = region_options,
        name = 'Region: '
    )
)

color = alt.condition(
    "(Select == 'All') || (datum.Region == Select)",
    alt.Color('Region:N', title = 'Region'),
    alt.value('lightgray')
)

chart = (
    alt.Chart(hpi_df)
    .mark_circle(size = 100, opacity = 0.85)
    .encode(
        x = alt.X("Life Expectancy (years):Q"),
        y = alt.Y('Happy Planet Index:Q'),
        color = color,
        tooltip = [
            'Country:N',
            'Region:N',
            alt.Tooltip('Wellbeing (0-10)', title = 'Wellbeing (0-10)'),
            alt.Tooltip('Life Expectancy (years):Q', title = 'Life Expectancy (years)'),
            alt.Tooltip('Happy Planet Index:Q', title = 'HPI'),
        ],
    )
    .add_params(region_pick)
    .properties(width = 600, height = 420, title = 'HPI - Value Based Selection by Region')
)

chart.save('ex01.html')

## Practice 12 – Problem 2
Plotly Express: animated CO₂ vs. GDP scatter plot (1990–2010).

In [ ]:
co2 = pd.read_csv('co2.csv')
gm = pd.read_csv('gapminder.csv')
gm = gm.rename(columns = {
    'gdpPercap': 'gdp',
    'Year': 'year',
    'continent': 'region',
    'Country': 'country'
})
df_gm = gm[['country', 'region']].dropna().drop_duplicates()

df_w_regions = pd.merge(
    co2, df_gm, on = 'country', how = 'inner'
)
year_col = [
    c for c in df_w_regions.columns if c.isdigit() and len(c) == 4
]

new_co2 = pd.melt(
    df_w_regions,
    id_vars = ['country', 'region'],
    value_vars = year_col,
    var_name = 'year',
    value_name = 'co2'
)
new_co2['year'] = pd.to_numeric(new_co2['year'], errors = 'coerce')
new_co2['co2'] = pd.to_numeric(new_co2['co2'], errors = 'coerce')

df_co2 = (
    new_co2.dropna(subset = ['year', 'co2'])
        .query('year >= 1990 & year <= 2010')
        .astype({'year': 'int64'})
        .sort_values(by = ['country', 'year'])
)

df_gdp = gm[['country', 'year', 'gdp']].copy()
df_gdp['year'] = pd.to_numeric(df_gdp['year'], errors = 'coerce')
df_gdp['gdp'] = pd.to_numeric(df_gdp['gdp'], errors = 'coerce')
df_gdp = (
    df_gdp.dropna(subset = ['year', 'gdp'])
        .drop_duplicates(['country', 'year'])
        .astype({'year': 'int64', 'gdp': 'float64'})
)

data = pd.merge(df_co2, df_gdp, on = ['country', 'year'], how = 'inner')
data = data.dropna(subset = ['gdp', 'co2'])
data = data[(data['gdp'] > 0) & (data['co2'] >= 0)]

if data.empty:
    raise ValueError("No data left afer cleaning (check inputs)")

xmin = max(1e-6, data['gdp'].min() * 0.95)
xmax = data['gdp'].max() * 1.05
ymin = max(0.0, data['co2'].min() * 0.95)
ymax = data['co2'].max() * 1.05

title_text = f'CO₂ Emiisions vs GDP (1990 - 2010)'

fig = px.scatter(
    data,
    x = 'gdp', y = 'co2',
    animation_frame = 'year',
    animation_group = 'country',
    color = 'region',
    hover_name = 'country',
    width = 1200, height = 600,
    log_x = True,
    size_max = 18,
    labels = {
        'gdp': 'GDP per capital',
        'co2': 'CO₂ per person',
        'region': 'Region',
        'country': 'Country',
        'year': 'Year'
    },
    title = title_text
)

fig.update_layout(legend_title_text = 'Region')

fig.write_html('ex02.html', include_plotlyjs = 'inline', full_html = 'True')
data.sort_values(['year', 'country','region'], kind = 'stable').to_csv(
    'ex02_clean.html', index = False
)


## Practice 13 – Problem 1
Altair: life expectancy vs. wellbeing scatter plot by region.

In [ ]:
alt.data_transformers.disable_max_rows()

hpi_url = ('http://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/hpi_data_countries.tsv')
hpi_df = pd.read_csv(hpi_url, sep = '\t')

chart = (
    alt.Chart(hpi_df, title = 'Life Expectancy vs. Wellbeing by Region')
    .mark_circle()
    .encode(
        x = alt.X('Life Expectancy (years):Q', title = 'Life Expectancy (years)'),
        y = alt.Y('Wellbeing (0-10):Q', title = 'Wellbeing'),
        color = 'Region:N',
        tooltip = [
            alt.Tooltip('Region:N', title = 'Region'),
            alt.Tooltip('Life Expectancy (years):Q', title = 'Life Expectancy'),
            alt.Tooltip('count():Q', title = '# Countries'),
            alt.Tooltip('Wellbeing (0-10):Q', title = 'Wellbeing')
        ]
    )
    .properties(width = 600, height = 400)
    .interactive()
)

output_html = 'ex01.html'
chart.save(output_html)

## Practice 13 – Problem 2
Altair: layered bar chart of mean wellbeing by region with a mean rule.

In [ ]:
region_order = ['Americas', 'Asia Pacific', 'Europe, Middle East and North Africa', 'Post-communist, Sub Saharan Africa']
bars_layer = (
    alt.Chart(hpi_df)
    .mark_bar()
    .encode(
        x = alt.X('Region:N', title = 'Region', sort = region_order),
        y = alt.Y('Wellbeing (0-10):Q', aggregate = 'mean', title = 'Mean Wellbeing'),
        tooltip = [
            alt.Tooltip('Region:N', title = 'Region'),
            alt.Tooltip('Wellbeing (0-10):Q', aggregate = 'mean', title = 'Mean Wellbeing'),
            alt.Tooltip('count():Q', title = '# Countries'),                    
        ]   
    )
    .properties(width = 500)
    .interactive()
)

line_layer = (
    alt.Chart(hpi_df)
    .mark_rule(color = 'red')
    .encode(
        y = alt.Y('Wellbeing (0-10):Q', aggregate = 'mean', title = 'Mean Wellbeing'),
        size = alt.SizeValue(3)
    )
)

#chart = alt.layer(bars_layer, line_layer, data = hpi_df).properties(title = 'Mean Happy Planet Index HPI by Region')
chart = (bars_layer + line_layer).properties(title = 'Mean Happy Planet Index HPI by Region')

output_html = 'ex02.html'
chart.save(output_html)


## Practice 14
Altair: linked charts, where selecting a category in the top-8 app categories bar chart filters the rating histogram.

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/TrainingByPackt/Interactive-Data-Visualization-with-Python/master/datasets/googleplaystore.csv'
gps_apps_df = pd.read_csv(url)
gps_apps_df = gps_apps_df.dropna(subset = ['Content Rating', 'Category', 'Rating'])
selector = alt.selection_point(fields = ['Category'])

top_cat = gps_apps_df['Category'].value_counts().nlargest(8).index
top_8 = gps_apps_df[gps_apps_df['Category'].isin(top_cat)]
bar_chart = (
    alt.Chart(top_8, title = 'Top 8 Apps Categories')
    .mark_bar()
    .encode(
        x = alt.X('Category:N', title = 'App Category', sort = '-y'),
        y = alt.Y('count():Q', title = 'Number of Apps'),
        color = alt.condition(selector, alt.value('steelblue'), alt.value('lightgrey'))
    )
    .properties(width = 250, height = 300)
    .add_params(selector)
)

histogram = (
    alt.Chart(gps_apps_df, title = 'App Rating Distribution')
    .mark_bar()
    .encode(
        x = alt.X('Rating:Q', bin = True, title = 'App Rating'),
        y = alt.Y('count():Q', title = 'Number of Apps'),
        color = alt.Color('Type:N', title = 'Free or Paid')
    )
    .transform_filter(selector)
)

linked = bar_chart | histogram
linked.save('gps_category_ratings.html')


## Practice 15
Bokeh time series: study session durations computed from start/end times, plotted by date, plus the longest session.

In [ ]:
data = {
    "Date": [
        "2026-10-19", "2026-10-20",
        "2026-10-21", "2026-10-22",
        "2026-10-23", "2026-10-24"
    ],
    "Start": [
        "09:00", "10:30", "13:00",
        "11:15", "14:00", "09:30"
    ],
    "End": [
        "10:30", "12:00", "15:30",
        "12:45", "16:15", "11:00"
    ]
}
df = pd.DataFrame(data)
df['Date'] = pd.to_datetime(df["Date"], format = 'mixed')

dt = df.copy()
start = pd.to_timedelta(dt['Start'] + ':00')
end = pd.to_timedelta(dt['End'] + ':00')
dt['Duration'] = ((end - start).dt.total_seconds() // 60).astype(int)

longest = dt.loc[dt["Duration"].idxmax()]


output_file('study_session.html')
p = figure(
    title = 'Study Session Duration Over Time',
    x_axis_label = 'Date',
    y_axis_label = 'Duration (minutes)',
    width = 800,
    height = 400
)
p.line(dt['Date'], dt['Duration'], line_width = 2)
p.scatter(dt['Date'], dt['Duration'], size = 8)
p.xaxis.formatter = DatetimeTickFormatter(days="%b %d", months="%b %d", hours="%b %d")
p.xaxis.ticker = DaysTicker(days=list(range(1, 32)))
show(p)

print(df.head())
print('---------------------------------')
print('Study Sessions:')
print(dt.head(), '\n\nLongest Study Session:')
print('Date:', longest['Date'].date())
print('Duration:', longest['Duration'], 'minutes')


## Practice 16 – Problem 1
Weekly resampling and date slicing of daily sales data.

In [ ]:
def to_datetime_mixed(data):
    try:
        return pd.to_datetime(data, format = 'mixed', errors = 'coerce')
    except TypeError:
        return pd.to_datetime(data, errors = 'coerce')

df = pd.DataFrame({
    "Date": [
    "2026-01-01", "2026-01-02",
    "2026-01-03", "2026-01-05",
    "2026-01-06", "2026-01-08",
    "2026-01-09", "2026-01-10"
    ],
    "Sales": [120, 140, 160, 110,
    180, 200, 150, 170]
})

df['Date'] = to_datetime_mixed(df['Date'])
df = df.set_index('Date').sort_index()
weekly_sales = df.resample('W-SUN').sum()

print(f'Original DataFrame:{df}')
print(f'\nSales from January 2 to January 6:\n{df.loc['2026-01-02':'2026-01-06'].head()}')
print(f'\nWeekly Total Sales:\n{weekly_sales}')

## Practice 16 – Problem 2
Altair bar chart of average online orders by hour.

In [ ]:
#problem 2
df = pd.DataFrame({
    "DateTime": [
    "2026-10-01 09:00",
    "2026-10-01 12:00",
    "2026-10-01 18:00",
    "2026-10-01 21:00",
    "2026-10-02 09:00",
    "2026-10-02 12:00",
    "2026-10-02 18:00",
    "2026-10-02 21:00"
    ],
    "Orders": [20, 50, 75, 40,
    30, 45, 85, 60]
})

df['DateTime'] = to_datetime_mixed(df['DateTime'])
df['hour'] = df['DateTime'].dt.hour

plot = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x = alt.X('hour:O'),
        y = alt.Y('mean(Orders):Q'),
        tooltip = ['hour', 'mean(Orders)']
    )
    .properties(title = 'Average Online Orders by Hour', width = 800, height = 400)
)

plot.save('Orders_by_Hour.html')


## Practice 17
Seaborn: group sales by day, compute totals and averages, and bar plot total sales by day.

In [ ]:
df = pd.DataFrame({
    "Day": ["Mon", "Mon", "Tue", "Tue",
    "Wed", "Wed", "Thu", "Thu"],
    "Sales": [100, 150, 180, 120,
    200, 160, 140, 190]
})

df = df.groupby('Day')
total_sales = df['Sales'].sum()
total_sales_df = df[['Sales']].sum()
average_sales = df['Sales'].mean()

ax = sns.barplot(
    data = total_sales_df,
    x = 'Day',
    y = 'Sales',
    order = ['Mon', 'Tue', 'Wed', 'Thu'],
    legend = False
)
ax.set_title('Total Sales by Day')
ax.set_xlabel("Day")
ax.set_ylabel("Total Sales")

print(f'Original Data:\n{df.head()}')
print(f'\nTotal Sales::\n{total_sales}')
print(f'\nAverage Sales::\n{average_sales}')
plt.show()


## Practice 18 – Problem 1
Temperature time series: upsample to daily frequency and linearly interpolate the missing values.

In [ ]:
dates = pd.to_datetime([
    "2026-01-01",
    "2026-01-03",
    "2026-01-05",
    "2026-01-07"
])
temperatures = pd.Series(
    [20, 24, 22, 28],
    index=dates
)

upsampled = temperatures.asfreq('D')
interpolate = upsampled.interpolate(method = 'linear')

plt.figure()
plt.plot(
    interpolate.index,
    interpolate,
    label = 'Interpolated Daily Data',
    color = 'orange'
)
plt.scatter(
    temperatures.index,
    temperatures,
    label = 'Orginal Data'
)
plt.title('Temperature: Original vs Interpolated')
plt.xlabel('Date')
plt.ylabel('Temperatures')
plt.legend()

## Practice 18 – Problem 2
Sales time series: compare `shift(1)` (value shift) with `shift(freq='1D')` (timestamp shift), then print results.

In [ ]:
#Question 2
dates = pd.date_range("2026-02-01", periods=5, freq="D")
sales = pd.Series(
    [100, 120, 110, 150, 140],
    index=dates
)

value_shift = sales.shift(1)
time_shift = sales.shift(freq = '1D')


plt.figure()
plt.plot(
    sales.index,
    sales,
    label = 'Original Sales',
    marker = 'o'
)
plt.plot(
    value_shift.index,
    value_shift,
    label = 'Value Shifted',
    marker = 's'
)
plt.plot(
    time_shift.index,
    time_shift,
    label = 'Timestamp Shifted',
    linestyle = '--',
    marker = '^'
)
plt.legend()
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Value Shifting vs. Timestamp Shifting')

print(f'Original Data:\n{temperatures}')
print(f'\nUpsampled Daily Data:\n{upsampled}')
print(f'\nInterpolated Daily Data:\n{interpolate}')

print(f'\n\nOriginal Sales:\n{sales}')
print(f'\nValue Shifted (shifted(1)):\n{value_shift}')
print(f"\nTimestamp Shifted (shifted(freq = '1D')):\n{time_shift}")
plt.show()

## Practice 19 – Problem 1
Time series: compute the autocorrelation of monthly sales with `acf` (lags 0–5) and show it as a stem plot.

In [ ]:
dates = pd.date_range(
    "2025-01-01", periods=12, freq="MS"
)
sales = pd.Series(
    [100, 120, 115, 130, 125, 140,
    135, 150, 145, 160, 155, 170],
    index=dates
)

autocorrelation = acf(sales, nlags = 5)
plt.figure()
plt.stem(
    autocorrelation
)
plt.xlabel('Lag (Months)')
plt.ylabel("Autocorrelation")
plt.title('Autocorrelation of Monthly Sales')
plt.show()

## Practice 19 – Problem 2
Bokeh line chart of monthly temperature, plus printed monthly sales and autocorrelation values by lag.

In [ ]:
months = [1, 2, 3, 4, 5, 6]
temperatures = [30, 35, 45, 55, 65, 75]

output_file('temperature_chart.html')
plot = figure(
    title = 'Monthly Temperature',
    x_axis_label = 'Month',
    y_axis_label = 'Temperature',
    tools = 'pan, wheel_zoom, box_zoom, reset, save'
)
plot.line(months, temperatures, line_width = 3, legend_label = 'Temperature')

print(f'Monthly Sales:\n{sales}')
print(f'\nAutocorrelation Valus')
for lag, value in enumerate(autocorrelation):
    print(f'Lag {lag} = {value:.4f}')

show(plot)